# Testing einfuehrung

<div style="display:flex;justify-content:space-between;align-items:center;background:#0d0d0f;border:1px solid #1e1e24;border-left:4px solid #4fc3f7;border-radius:6px;padding:clamp(1rem,2.5vw,1.8rem) clamp(1.2rem,3vw,2.4rem);margin-bottom:2rem;position:relative;overflow:hidden;box-shadow:0 4px 32px rgba(0,0,0,0.5);font-family:'Segoe UI',sans-serif;">
<div style="position:absolute;top:0;left:0;right:0;bottom:0;background:radial-gradient(ellipse at 0% 50%,rgba(79,195,247,0.07) 0%,transparent 60%);pointer-events:none;"></div>
<div style="display:flex;flex-direction:column;gap:0.3rem;">
<p style="font-size:clamp(1.3rem,3.5vw,2.4rem);color:#f0f0f5;margin:0;line-height:1.1;font-weight:700;letter-spacing:-0.01em;">Testen: Warum, was und wie systematisch?</p>
<p style="font-size:clamp(0.75rem,1.6vw,1rem);color:#7a7a90;margin:0;letter-spacing:0.04em;font-weight:300;">Development Expert Python / PCAP &nbsp;|&nbsp; Kapitel 10: Testing &nbsp;|&nbsp; Notebook 10a</p>
</div>
</div>

**Legende**

> **[Kursinhalt]** Dieses Notebook ist kein PCAP-Pruefungsinhalt

---

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
1. Das Problem: Code den man nicht vertraut
</span>
</div>

Wir haben in Kapitel 5 bereits `unittest` kennengelernt -- ein Modul um automatische Tests zu schreiben. Warum kommt Testing jetzt nochmal als eigenes Kapitel?

Weil es einen Unterschied gibt zwischen *Tests schreiben koennen* und *systematisch testen*.

Schauen wir uns ein konkretes Szenario an. Wir haben ein Kampfsystem fuer ein Rollenspiel entwickelt -- `combat.py`. Die Funktionen sehen vernuenftig aus, die Grundfaelle funktionieren. Aber stecken da Fehler drin?

Lass uns manuell pruefen:

In [ ]:
from Aetheria_Game.combat import berechne_schaden, heilen, ist_besiegt, trefferchance, kritischer_treffer, statuseffekt_anwenden

# Typische Faelle -- sieht alles gut aus
print(berechne_schaden(50, 20))    # Erwartet: 30
print(heilen(60, 100, 20))         # Erwartet: 80
print(ist_besiegt(0))              # Erwartet: True
print(trefferchance(10, 8))        # Erwartet: 60
print(kritischer_treffer(30))      # Erwartet: 45
print(statuseffekt_anwenden(20, 'vergiftet'))  # Erwartet: 30

Alles sieht korrekt aus. Sechs Aufrufe, sechs erwartete Ergebnisse. Wer wuerde hier einen Fehler vermuten?

Aber was passiert in diesen Situationen:

In [ ]:
# Was passiert wenn Verteidigung den Angriff uebersteigt?
print(berechne_schaden(10, 50))    # Erwartet: 1 (Mindestschaden)
                                   # Erhalten: ???

# Was passiert wenn Heilung ueber die maximalen HP geht?
print(heilen(90, 100, 50))         # Erwartet: 100 (Cap bei max_hp)
                                   # Erhalten: ???

# Was passiert bei negativen HP?
print(ist_besiegt(-10))            # Erwartet: True (tot ist tot)
                                   # Erhalten: ???

# Was wenn Geschwindigkeit sehr hoch ist?
print(trefferchance(20, 1))        # Erwartet: max. 95%
                                   # Erhalten: ???

# Was gibt kritischer_treffer zurueck?
krit = kritischer_treffer(30)
print(type(krit), krit)            # Erwartet: int
                                   # Erhalten: ????

# Was passiert bei unbekanntem Statuseffekt?
print(statuseffekt_anwenden(20, 'eingefroren'))  # Erwartet: ValueError
                                                  # Erhalten: ????

Sechs Bugs in sechs Funktionen -- und alle waren im normalen Gebrauch unsichtbar. Das ist der Punkt.

Manuelles Testen prueft immer nur die Faelle die man sich gerade vorstellt. Systematisches Testen prueft **alle** relevanten Faelle -- auch die die man vergisst.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
2. Das Kampfsystem -- combat.py
</span>
</div>

Das Modul `combat.py` begleitet uns durch alle Testing-Notebooks. Es ist ein vereinfachtes aber realistisches Kampfsystem -- so wie es in einem echten Rollenspiel vorkommen koennte.

Sechs Funktionen, jede mit einer klaren Aufgabe und einem klar definierten Verhalten. Und jede mit einem eingebauten Bug der beim normalen Gebrauch nicht auffaellt -- aber beim systematischen Testen sofort.

Schauen wir uns jede Funktion genau an:

In [2]:
# berechne_schaden(angriff, verteidigung)
# Nettoschaden = Angriff - Verteidigung
# Mindestschaden: 1 -- ein Treffer tut IMMER weh
# Negative Werte: nicht erlaubt -> ValueError

from Aetheria_Game.combat import berechne_schaden

print(berechne_schaden(50, 20))   # 30  -- Normalfall
print(berechne_schaden(20, 20))   # ?   -- gleiche Werte
print(berechne_schaden(10, 50))   # ?   -- Verteidigung > Angriff
print(berechne_schaden(1, 0))     # ?   -- minimaler Angriff

30
0
0
1


In [ ]:
# trefferchance(angreifer_geschwindigkeit, ziel_geschwindigkeit)
# Formel: 50 + (Angreifer - Ziel) * 5
# Minimum: 10%  -- immer eine Chance zu treffen
# Maximum: 95%  -- immer eine Chance zu verfehlen
# Geschwindigkeit muss > 0 sein

from Aetheria_Game.combat import trefferchance

print(trefferchance(10, 10))   # 50  -- gleich schnell
print(trefferchance(10, 8))    # 60  -- etwas schneller
print(trefferchance(1, 10))    # ?   -- viel langsamer
print(trefferchance(20, 1))    # ?   -- viel schneller -- ueber 95?

In [ ]:
# kritischer_treffer(schaden, krit_faktor=1.5)
# Schaden * krit_faktor, abgerundet auf int
# krit_faktor muss >= 1.0 sein

from Aetheria_Game.combat import kritischer_treffer

krit = kritischer_treffer(30)
print(krit, type(krit))     # 45 int -- oder float?

krit2 = kritischer_treffer(30, krit_faktor=2.0)
print(krit2, type(krit2))   # 60 int -- oder float?

In [ ]:
# heilen(aktuelle_hp, max_hp, heilmenge)
# HP erhoeht sich um heilmenge
# Aber nie ueber max_hp
# Toter Held (hp <= 0) kann nicht geheilt werden
# Negative heilmenge nicht erlaubt

from Aetheria_Game.combat import heilen

print(heilen(60, 100, 20))    # 80  -- Normalfall
print(heilen(90, 100, 50))    # ?   -- wuerde ueber max_hp gehen
print(heilen(100, 100, 10))   # ?   -- schon voll
print(heilen(0, 100, 50))     # ?   -- Held ist tot

In [ ]:
# ist_besiegt(hp)
# True wenn hp <= 0

from Aetheria_Game.combat import ist_besiegt

print(ist_besiegt(100))   # False
print(ist_besiegt(1))     # False -- noch knapp am Leben
print(ist_besiegt(0))     # True
print(ist_besiegt(-10))   # ?     -- negative HP durch starken Treffer

In [ ]:
# statuseffekt_anwenden(schaden, effekt)
# Erlaubte Effekte: 'normal', 'vergiftet', 'geschwaeht', 'verstaerkt'
# Unbekannter Effekt: ValueError

from Aetheria_Game.combat import statuseffekt_anwenden

print(statuseffekt_anwenden(20, 'normal'))      # 20
print(statuseffekt_anwenden(20, 'vergiftet'))   # 30
print(statuseffekt_anwenden(20, 'geschwaeht'))  # 15
print(statuseffekt_anwenden(20, 'verstaerkt'))  # 40

# Unbekannter Effekt -- was passiert?
try:
    print(statuseffekt_anwenden(20, 'eingefroren'))
except ValueError as e:
    print(f'ValueError: {e}')

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
3. Aequivalenzklassen -- nicht jeden Wert einzeln testen
</span>
</div>

Eine Funktion wie `heilen(aktuelle_hp, max_hp, heilmenge)` hat theoretisch unendlich viele moegliche Eingaben. Man kann nicht jeden Wert testen. Aber man muss es auch nicht.

Die Idee hinter **Aequivalenzklassen**: Eingaben die sich gleich verhalten werden zu einer Gruppe zusammengefasst. Aus jeder Gruppe genuegt ein einziger Repraesentant.

Fuer `heilen()` sehen die Klassen so aus:

| Klasse | Beschreibung | Beispielwert | Erwartetes Verhalten |
|--------|-------------|--------------|---------------------|
| Gueltige HP (lebendig) | 1 bis max_hp | `aktuelle_hp=60` | Heilung wird angewendet |
| Grenzfall: voll | HP == max_hp | `aktuelle_hp=100` | Keine Aenderung |
| Toter Held | HP <= 0 | `aktuelle_hp=0` | Keine Heilung |
| Gueltige Heilmenge | heilmenge > 0 | `heilmenge=20` | Wird addiert |
| Ungueltige Heilmenge | heilmenge <= 0 | `heilmenge=0` | ValueError |
| Heilung bleibt im Bereich | aktuelle_hp + heilmenge <= max_hp | `60 + 20 = 80` | Exakt addiert |
| Heilung uebersteigt Maximum | aktuelle_hp + heilmenge > max_hp | `90 + 50 = 140` | Wird auf max_hp gekappt |

Sieben Klassen -- sieben Tests. Das deckt das gesamte Verhalten der Funktion ab.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
4. Grenzwertanalyse -- genau dort wo Fehler lauern
</span>
</div>

Aequivalenzklassen sagen *welche Gruppen* es gibt. Die **Grenzwertanalyse** sagt *welche Werte innerhalb dieser Gruppen* besonders interessant sind.

Die Erfahrung zeigt: Bugs verstecken sich fast immer an den Grenzen. Nicht bei `hp=50` -- sondern bei `hp=0`, `hp=1`, `hp=max_hp-1`, `hp=max_hp`.

Die Regel: An jeder Grenze testet man drei Werte -- **einen darunter**, **die Grenze selbst**, **einen darueber**.

Fuer `heilen()` mit `max_hp=100`:

```
Grenze: aktuelle_hp = 0 (tot/lebendig)
  darunter:  hp = -1  --> kein Effekt (tot)
  Grenze:    hp =  0  --> kein Effekt (tot)
  darueber:  hp =  1  --> Heilung wirkt (lebendig)

Grenze: Heilung trifft max_hp
  darunter:  90 + 9  = 99  --> 99   (unter max_hp)
  Grenze:    90 + 10 = 100 --> 100  (genau max_hp)
  darueber:  90 + 11 = 101 --> 100  (wird gekappt)
```

Diese Grenzwerte sind es die Bugs aufdecken -- nicht die mittleren Werte.

In [ ]:
from Aetheria_Game.combat import heilen, ist_besiegt, berechne_schaden

# Grenzwerte fuer heilen() manuell pruefen
print('--- Grenze: tot/lebendig ---')
print(f'hp=-1: {heilen(-1, 100, 20)}')   # kein Effekt
print(f'hp= 0: {heilen( 0, 100, 20)}')   # kein Effekt
print(f'hp= 1: {heilen( 1, 100, 20)}')   # 21

print()
print('--- Grenze: max_hp-Cap ---')
print(f'90 + 9  = {heilen(90, 100,  9)}')   # 99
print(f'90 + 10 = {heilen(90, 100, 10)}')   # 100
print(f'90 + 11 = {heilen(90, 100, 11)}')   # 100 -- wird gekappt?

print()
print('--- Grenze: Mindestschaden ---')
print(f'angriff=verteidigung:   {berechne_schaden(20, 20)}')   # 1
print(f'verteidigung > angriff: {berechne_schaden(10, 50)}')   # 1
print(f'verteidigung = 0:       {berechne_schaden(1, 0)}')     # 1

print()
print('--- Grenze: ist_besiegt ---')
print(f'hp= 1: {ist_besiegt( 1)}')   # False
print(f'hp= 0: {ist_besiegt( 0)}')   # True
print(f'hp=-1: {ist_besiegt(-1)}')   # True -- oder nicht?

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
5. Was wir gefunden haben -- und was als naechstes kommt
</span>
</div>

Allein durch manuelles Pruefen der Grenzwerte haben wir sechs Bugs gefunden:

| Funktion | Bug | Auswirkung im Spiel |
|----------|-----|--------------------|
| `berechne_schaden` | Mindestschaden ist 0 statt 1 | Angriffe koennen wirkungslos sein |
| `trefferchance` | Kein Maximum bei 95% | Held kann unfehlbar werden |
| `kritischer_treffer` | Gibt float zurueck statt int | Folgefehler bei HP-Berechnungen |
| `heilen` | Kein Cap bei max_hp | HP koennen ueber Maximum steigen |
| `ist_besiegt` | Negative HP gelten nicht als besiegt | Held kaempft mit -100 HP weiter |
| `statuseffekt_anwenden` | Unbekannte Effekte werden ignoriert | Tippfehler bleiben unbemerkt |

Das Problem mit manuellem Testen: Wir haben das jetzt einmal geprueft. Wenn jemand morgen `combat.py` veraendert und dabei versehentlich einen neuen Bug einbaut, merken wir es nicht -- ausser wir pruefen alles manuell erneut.

**Automatisierte Tests** laufen bei jeder Aenderung. Sie pruefen immer dieselben Grenzwerte. Sie vergessen nichts.

In Notebook **10b** schreiben wir genau diese Tests mit **pytest**.

---

**Zusammenfassung**

| Konzept | Erklaerung |
|---------|------------|
| Manuelles Testen | Prueft nur was man sich gerade vorstellt -- vergisst Grenzfaelle |
| Aequivalenzklassen | Eingaben mit gleichem Verhalten zu Gruppen zusammenfassen -- ein Test pro Gruppe genuegt |
| Grenzwertanalyse | Bugs verstecken sich an den Grenzen -- darunter, die Grenze selbst, darueber |
| Automatisierte Tests | Laufen bei jeder Aenderung -- vergessen nichts |

**Die Bugs in combat.py -- Uebersicht:**

```python
berechne_schaden:        max(0, schaden)    # sollte max(1, schaden) sein
trefferchance:           max(10, chance)    # min(95, ...) fehlt
kritischer_treffer:      schaden * faktor   # sollte int(...) sein
heilen:                  hp + heilmenge     # max_hp-Cap fehlt
ist_besiegt:             hp == 0            # sollte hp <= 0 sein
statuseffekt_anwenden:   return schaden     # sollte ValueError werfen
```